# 01 · EDA exploratorio y propuesta de taxonomía
Clustering de los nombres del catálogo completo para descubrir las familias de
productos escondidas en «General». No es parte de la nota: sirve para definir las
categorías que se van a etiquetar.

In [1]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import silhouette_score

from catalogo import SEED
from catalogo.data import find_root, load_catalog

ROOT = find_root()
df = load_catalog(ROOT / "data/raw/productos.csv")
len(df)

10777

In [2]:
vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, sublinear_tf=True)
X = vec.fit_transform(df["nombre"])
X.shape

(10777, 21551)

In [3]:
ks = range(6, 21, 2)
sil = {}
for k in ks:
    km = KMeans(n_clusters=k, n_init=5, random_state=SEED).fit(X)
    sil[k] = silhouette_score(X, km.labels_, sample_size=3000, random_state=SEED)
pd.Series(sil, name="silhouette")

6     0.020824
8     0.023029
10    0.026006
12    0.025040
14    0.030653
16    0.035559
18    0.033541
20    0.040081
Name: silhouette, dtype: float64

In [4]:
K = max(sil, key=sil.get)
km = KMeans(n_clusters=K, n_init=10, random_state=SEED).fit(X)
df["cluster"] = km.labels_
terms = np.array(vec.get_feature_names_out())
for c in range(K):
    top = terms[km.cluster_centers_[c].argsort()[::-1][:12]]
    sample = df.loc[df["cluster"] == c, "nombre"].sample(
        min(8, (df["cluster"] == c).sum()), random_state=SEED
    )
    print(f"\n=== Clúster {c} · {(df['cluster'] == c).sum()} productos")
    print("n-gramas:", ", ".join(top))
    print("\n".join("  - " + s for s in sample))


=== Clúster 0 · 120 productos
n-gramas: ely , ely, ly ,  itel, itel,  ite, tely , tely, itely, kuul , uul , kuu
  - ITELY AQUARELY 7NF/7.01 RUBIO NATURAL FRIO
  - ITELY AQUARELY 7RF/7.5 RUBIO ROJO INTENSO
  - ITELY AQUARELY 10N/10.0 RUBIO ULTRA CLARO
  - ITELY AQUARELY 8NF/8.01 RUBIO CLARO NATURAL FRIO
  - ITELY AQUARELY 6C/6.1 RUBIO OSCURO CENIZA
  - KUUL FUNNY COLORS TINTE AZUL CELESTE 90 ML
  - KUUL FUNNY COLORS TINTE ROJO ROSADO 90 ML
  - ITELY AQUARELY 1C/1.1 NEGRO AZULADO 100ML

=== Clúster 1 · 187 productos
n-gramas: polvo, olvo , olvo, lvo, lvo , polv, olv, vo ,  polv, pol,  pol,  po
  - GOLD´MAR POLVO DECOLORANTE 9+ 20G
  - NAI POLVO COMP 0
  - DUO DE ESPONJAS EN POLVO Y BASE LIQUIDA
  - POLVO POWDER STAY BEAUTY CREATIONS 12.0
  - POLVO TRASLUCIDO USHAS
  - B.CREAT.POLVO FSP11.0 (7G/025OZ)
  - MN.POLVO ACRIL.1OZ PINK
  - POLVO SUELTO LA COLORS

=== Clúster 2 · 4279 productos
n-gramas:  de, de ,  de , as ,  pe, ado, dor, or , es ,  ma, er , el 
  - EOC SHINE 2025 SPREMITUBO
  